# Dynamic Sessions — Python REPL & Shell Tools

### Introduction

[Azure Container Apps dynamic sessions](https://learn.microsoft.com/azure/container-apps/sessions) provide fast, fully isolated, sandboxed environments that an agent can use to run arbitrary code safely. Each session runs in its own Hyper-V isolated container, so untrusted LLM-generated code can execute without putting your infrastructure at risk.

In this section we give a LangChain agent two powerful tools backed by dynamic sessions:

- a **Python REPL** tool (`SessionsPythonREPLTool`) to execute Python code, and
- a **Shell / Bash** tool (`SessionsBashTool`) to run shell commands.

We also show how to upload and download files to and from a session.

### Prerequisites

- Completion of steps in `010_langchain_basics.ipynb` where we deployed an Azure Foundry with an LLM model.
- A dynamic session pool (Python and Shell) deployed on Azure Container Apps (see the `infra` folder).
- You must be logged in with the Azure CLI (`az login`) so the notebook can acquire tokens for the session pool.

### Getting the LLM Endpoint and API Key

To use the LLM model with `langchain`, we first need to retrieve the endpoint, API key and the session pool management endpoints from the Terraform outputs. You can do this by running the following command.


In [1]:
foundry_endpoint = ! terraform -chdir=./infra output -raw foundry_endpoint
foundry_endpoint = foundry_endpoint.n
print("Foundry Endpoint:", foundry_endpoint)

foundry_api_key = ! terraform -chdir=./infra output -raw foundry_api_key
foundry_api_key = foundry_api_key.n
print("Foundry API Key:", f"{foundry_api_key[-10:]}...")  # Print only the last 10 characters for security

llm_model_deployment_name = ! terraform -chdir=./infra output -raw llm_model_deployment_name
llm_model_deployment_name = llm_model_deployment_name.n
print("LLM Model Deployment Name (ChatGPT):", llm_model_deployment_name)

sessionpool_management_endpoint_python = ! terraform -chdir=./infra output -raw sessionpool_management_endpoint_python
sessionpool_management_endpoint_python = sessionpool_management_endpoint_python.n
print("Session Pool Management Endpoint:", sessionpool_management_endpoint_python)

sessionpool_mcp_endpoint_python = ! terraform -chdir=./infra output -raw sessionpool_mcp_endpoint_python
sessionpool_mcp_endpoint_python = sessionpool_mcp_endpoint_python.n
print("MCP Session Pool Endpoint:", sessionpool_mcp_endpoint_python)

sessionpool_management_endpoint_shell = ! terraform -chdir=./infra output -raw sessionpool_management_endpoint_shell
sessionpool_management_endpoint_shell = sessionpool_management_endpoint_shell.n
print("Session Pool Management Endpoint (Shell):", sessionpool_management_endpoint_shell)

sessionpool_mcp_endpoint_shell = ! terraform -chdir=./infra output -raw sessionpool_mcp_endpoint_shell
sessionpool_mcp_endpoint_shell = sessionpool_mcp_endpoint_shell.n
print("MCP Session Pool Endpoint (Shell):", sessionpool_mcp_endpoint_shell)

Foundry Endpoint: https://foundry-400.cognitiveservices.azure.com/
Foundry API Key: AAACOGYQ3t...
LLM Model Deployment Name (ChatGPT): gpt-5.4
Session Pool Management Endpoint: https://swedencentral.dynamicsessions.io/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-ai-agents-langchain-400/sessionPools/acasessionpool-python
MCP Session Pool Endpoint: https://swedencentral.dynamicsessions.io/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-ai-agents-langchain-400/sessionPools/acasessionpool-python/mcp
Session Pool Management Endpoint (Shell): https://swedencentral.dynamicsessions.io/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-ai-agents-langchain-400/sessionPools/acasessionpool-shell
MCP Session Pool Endpoint (Shell): https://swedencentral.dynamicsessions.io/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-ai-agents-langchain-400/sessionPools/acasessionpool-shell/mcp


### 1. Simple Chat - No Tools

`OpenAI` API defined the standard spec for calling LLM models. Most of the inference tools and frameworks are built on top of this API providing a kind of wrapper. LangChain is one of them.

Create a `ChatOpenAI` model pointing at the Azure Foundry OpenAI-compatible endpoint and send a basic message. Note how we can enable streaming responses for real-time output. The commented block shows how the same code can point at an LLM hosted on Azure Container Apps with GPU instead.


In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

model = ChatOpenAI(
    base_url=f"{foundry_endpoint}/openai/v1",
    api_key=foundry_api_key,
    model=llm_model_deployment_name,
    streaming=True,
    max_completion_tokens=512
)

response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

I’m ChatGPT, an AI assistant created by OpenAI.

I can help with things like:
- answering questions
- explaining concepts
- writing and editing
- brainstorming ideas
- summarizing information
- coding help
- math and logic
- planning and organizing

A few useful things to know about me:
- I don’t have feelings, beliefs, or personal experiences.
- I generate responses based on patterns in data I was trained on.
- I can be very helpful, but I can also make mistakes, so it’s good to verify important information.
- I don’t automatically know private or real-time information unless you provide it or the system I’m connected to gives me access.

You can talk to me casually, ask for short or detailed answers, or have me adapt to a style you prefer.

If you want, I can also tell you:
- what I’m good at
- what my limitations are
- how to get better results from me

### 2. Python REPL Session Tool

Create a `SessionsPythonREPLTool` bound to the Python session pool. The tool acquires an access token via the Azure CLI credential and lets the agent execute Python code in a sandboxed session. We then build a ReAct agent and ask it a question that requires running code.


In [4]:
from langchain.agents import create_agent
from langchain_azure_dynamic_sessions.tools import SessionsPythonREPLTool
from azure.identity import AzureCliCredential

credential = AzureCliCredential()

def access_token_provider():
    token = credential.get_token("https://dynamicsessions.io/.default")
    return token.token

# get the management endpoint from the session pool in the Azure portal
toolPythonSession = SessionsPythonREPLTool(
    pool_management_endpoint=sessionpool_management_endpoint_python,
    access_token_provider=access_token_provider,
)

agent = create_agent(model=model, tools=[toolPythonSession])

async for step in agent.astream(
    {"messages": [{"role": "user", "content": "What is the current time in Tunisia and France ? Use the python REPL tool to get the answer."}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the current time in Tunisia and France ? Use the python REPL tool to get the answer.
================================== Ai Message ==================================
Tool Calls:
  Python_REPL (call_fx3XEE7Ms0fWHDhDhchb4hW7)
 Call ID: call_fx3XEE7Ms0fWHDhDhchb4hW7
  Args:
    python_code: from datetime import datetime
from zoneinfo import ZoneInfo
now_tunisia = datetime.now(ZoneInfo('Africa/Tunis'))
now_france = datetime.now(ZoneInfo('Europe/Paris'))
print('Tunisia:', now_tunisia.strftime('%Y-%m-%d %H:%M:%S %Z%z'))
print('France:', now_france.strftime('%Y-%m-%d %H:%M:%S %Z%z'))
================================= Tool Message =================================
Name: Python_REPL

{
  "result": "",
  "stdout": "Tunisia: 2026-06-06 21:05:41 CET+0100\nFrance: 2026-06-06 22:05:41 CEST+0200\n",
  "stderr": ""
}
================================== Ai Message ==================================

Current local t

### 3. Upload Files to the Python Session

You can upload files into a session and reference them from the executed code. Here we upload a JSON file and run code that reads it back and computes a value.


In [5]:
import io
import json

data = {"important_data": [1, 10, -1541]}
binary_io = io.BytesIO(json.dumps(data).encode("ascii"))

upload_metadata = toolPythonSession.upload_file(
    data=binary_io, remote_file_path="important_data.json"
)

code = f"""
import json

with open("{upload_metadata.full_path}") as f:
    data = json.load(f)

sum(data['important_data'])
"""

toolPythonSession.execute(code)

{'$id': '2',
 'status': 'Success',
 'stdout': '',
 'stderr': '',
 'result': -1530,
 'executionTimeInMilliseconds': 7}

### 4. Bash / Shell Session Tool

The `SessionsBashTool` runs shell commands inside the sandboxed session. The cells below explore the environment — printing environment variables, inspecting the code interpreter config, and checking the outbound IP — to demonstrate the isolation provided by dynamic sessions.


In [7]:
from langchain_azure_dynamic_sessions.tools import SessionsBashTool

# get the management endpoint from the session pool in the Azure portal
toolBashSession = SessionsBashTool(
    pool_management_endpoint=sessionpool_management_endpoint_shell,
    access_token_provider=access_token_provider,
)

response = toolBashSession.execute("echo Hello from the session!")

print(json.dumps(response, indent=4))

{
    "identifier": "792c15f6-ce7c-4ef9-86c2-c4aac30d7204",
    "status": "0",
    "result": {
        "stdout": "Hello from the session!\n",
        "stderr": "",
        "executionTimeInMilliseconds": 2
    }
}


Let's try to run a shell command to check the environment variables of the session.

In [8]:
response = toolBashSession.execute("printenv")

print(json.dumps(response, indent=4))

{
    "identifier": "792c15f6-ce7c-4ef9-86c2-c4aac30d7204",
    "status": "0",
    "result": {
        "stdout": "PWD=/mnt/data\nENABLE_EGRESS=false\n_=/usr/bin/printenv\nHOME=/root\nCODE_INTERPRETER_CONFIG_FILE_PATH=/app/codeInterpreterConf.yaml\nSHLVL=1\nPATH=/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin\n",
        "stderr": "",
        "executionTimeInMilliseconds": 3
    }
}


Let's check the configuration file for the code interpreter.

In [9]:
response = toolBashSession.execute("cat /app/codeInterpreterConf.yaml")

print(json.dumps(response, indent=4))

{
    "identifier": "792c15f6-ce7c-4ef9-86c2-c4aac30d7204",
    "status": "0",
    "result": {
        "stdout": "kind: CodeInterpreterConfig\napiVersion: v1\nmetadata:\n  name: code-interpreter\nspec:\n  logging:\n    logLevel: 1 # 0: debug, 1: info, 2: warn, 3: error\n    useISO8601DateFormat: false\n  kernel:\n    enabled: false\n    isFirstParty: true\n    port: 6003\n\n  healthProbe: # It will respond back /health endpoint with 200 status code if the container is healthy\n    initialDelaySeconds: 0\n\n  fileOperation:\n    maxUploadFileSizeLimitInMBs: 250\n    enableFullPathAccess: true\n",
        "stderr": "",
        "executionTimeInMilliseconds": 2
    }
}


Let's try to run a shell command to check the outbound IP address of the session.

In [10]:
response = toolBashSession.execute("curl ifconfig.me/ip")

print(json.dumps(response, indent=4))

{
    "identifier": "792c15f6-ce7c-4ef9-86c2-c4aac30d7204",
    "status": "0",
    "result": {
        "stdout": "74.241.203.7",
        "stderr": "  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current\n                                 Dload  Upload   Total   Spent    Left  Speed\n\r  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0\r100    12  100    12    0     0     63      0 --:--:-- --:--:-- --:--:--    64\n",
        "executionTimeInMilliseconds": 256
    }
}


### 5. Agent with the Shell Tool

Now we hand the shell tool to an agent. The agent can install packages, write files, and run programs entirely inside the session — for example spinning up a Flask app or fetching and plotting stock data.


In [12]:
agent = create_agent(model=model, tools=[toolBashSession])

async for step in agent.astream(
    {"messages": [{
        "role": "user", 
        "content": """
            Create a hello world flask app in a new remote environment. 
            Then send a request to the app to show that it works.
        """
        }]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================


            Create a hello world flask app in a new remote environment. 
            Then send a request to the app to show that it works.
        
================================== Ai Message ==================================
Tool Calls:
  Bash (call_eiNraO4TlVxSlAL34132JrEX)
 Call ID: call_eiNraO4TlVxSlAL34132JrEX
  Args:
    bash_command: set -e
mkdir -p /tmp/flask_hello_app && cd /tmp/flask_hello_app
python3 -m venv .venv
source .venv/bin/activate
pip install --quiet flask
cat > app.py <<'PY'
from flask import Flask
app = Flask(__name__)

@app.route('/')
def hello():
    return 'Hello, World!'

if __name__ == '__main__':
    app.run(host='127.0.0.1', port=5000)
PY
nohup .venv/bin/python app.py > server.log 2>&1 &
SERVER_PID=$!
sleep 3
curl -s http://127.0.0.1:5000/
printf "\nPID:%s\n" "$SERVER_PID"

    kwargs: {'timeout': 120000}
================================= Tool Message =====================

In [34]:
agent = create_agent(model=model, tools=[toolBashSession])

async for step in agent.astream(
    {"messages": [{"role": "user", 
                   "content": """
                        Get the history of the Microsoft stock price for the last 5 years and plot it. 
                        Install the required packages in the session if needed.
                        save the chart as `msft_5y.png`.
                   """}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================


                        Get the history of the Microsoft stock price for the last 5 years and plot it. 
                        Install the required packages in the session if needed.
                        save the chart as `msft_5y.png`.
                   
================================== Ai Message ==================================
Tool Calls:
  Bash (call_qyVZ81moJoKBjt48prSYhchB)
 Call ID: call_qyVZ81moJoKBjt48prSYhchB
  Args:
    bash_command: python - <<'PY'
import importlib.util, subprocess, sys, os
pkgs = ['yfinance','pandas','matplotlib']
for p in pkgs:
    mod = p.split('[')[0]
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', p])

import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

symbol = 'MSFT'
# 5 years of daily history
hist = yf.Ticker(symbol).history(period='5y', auto_adjust=False)
if hist.emp

### 6. Download Files from the Session

Files created inside a session — such as the chart generated above — can be listed and downloaded back to your local machine. Here we download the generated chart and display it in the notebook.


In [35]:
# upload a file to the session
# toolBashSession.upload_file(local_file_path="./rg.tf", remote_file_path="/mnt/user/rg.tf")

# list files in the session
files = toolBashSession.list_files()

# print files in the session
print(files)

# # download a file from the session
toolBashSession.download_file(remote_file_path="msft_5y.png", local_file_path="./output/msft_5y.png")

# view the image in the notebook
from PIL import Image
image = Image.open("./output/msft_5y.png")
image.show()

[RemoteFileMetadata(filename='msft.csv', size_in_bytes=23), RemoteFileMetadata(filename='msft_5y.png', size_in_bytes=115703), RemoteFileMetadata(filename='msft_stooq.csv', size_in_bytes=796)]


### More Resources

- [Azure Container Apps dynamic sessions](https://learn.microsoft.com/azure/container-apps/sessions) — fast, isolated, sandboxed code execution
- [Code interpreter sessions](https://learn.microsoft.com/azure/container-apps/sessions-code-interpreter) — run untrusted LLM-generated code safely
- [langchain-azure-dynamic-sessions](https://pypi.org/project/langchain-azure-dynamic-sessions/) — the LangChain integration used in this notebook
- [LangGraph ReAct Agent](https://python.langchain.com/docs/tutorials/agents/)
- [ChatOpenAI with custom endpoints](https://python.langchain.com/api_reference/openai/chat_models/langchain_openai.chat_models.base.ChatOpenAI.html)
